# 🚀 Target-Aware Query Attention & Component Interaction
### Kaggle Interactive Runner & In-Depth Pipeline Inspection

This notebook clones the repository, loads the native Python modules directly from `src/`, and provides **interactive inspection** of data loading, subtoken span alignments, model tensor shapes, and full 2-phase training.

In [ ]:
# =============================================================================
# 1. Git Clone & Module Import Setup
# =============================================================================
import os
import sys
from pathlib import Path

# Update repository URL if cloning to a new environment
REPO_URL = "https://github.com/nam722006/compositionality.git"
REPO_DIR = "/kaggle/working/compositionality"

if not os.path.exists(REPO_DIR) and not os.path.exists("src"): 
    print(f"📥 Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
elif os.path.exists(REPO_DIR):
    print(f"📁 Setting workspace to: {REPO_DIR}")
    os.chdir(REPO_DIR)
    !git pull

# Ensure project root is in sys.path
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"📍 Active Directory: {os.getcwd()}")

# Install requirements
if os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt
else:
    !pip install -q torch transformers datasets scipy pandas numpy matplotlib scikit-learn

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 PyTorch: {torch.__version__} | Device: {device} | CUDA Available: {torch.cuda.is_available()}")

In [ ]:
# =============================================================================
# 2. Load Configuration from Project Configs
# =============================================================================
from src.config import Config
from src.utils import set_seed, get_logger

config_path = "config/target_aware_query_attention.json"
if os.path.exists(config_path):
    cfg = Config.from_json(config_path)
    print(f"📋 Loaded configuration from {config_path}")
else:
    cfg = Config()
    print("📋 Using default Config dataclass")

# Configure hyper-parameters for Kaggle GPU
cfg.batch_size = 16
cfg.freeze_epochs = 3
cfg.unfreeze_epochs = 6
cfg.warmup_ratio = 0.1
cfg.targets = ['mod', 'head', 'pv']

set_seed(cfg.seed)
print(f"🔹 Backbone: {cfg.backbone}")
print(f"🔹 Batch Size: {cfg.batch_size} | Freeze Epochs: {cfg.freeze_epochs} | Unfreeze Epochs: {cfg.unfreeze_epochs}")
print(f"🔹 Active Targets: {cfg.targets}")

In [ ]:
# =============================================================================
# 3. Data Inspection: Load Dataset Rows & View Statistics
# =============================================================================
import pandas as pd
from src.data import load_labeled, expand_targets

# Attempt loading labeled datasets
try:
    raw_rows = load_labeled(cfg)
    df = pd.DataFrame(raw_rows)
    print(f"✅ Successfully loaded {len(raw_rows)} total rows from dataset files!")
    print(f"\n📊 Dataset Distribution:")
    print(f"  • Languages: {dict(df['lang'].value_counts())}")
    print(f"  • Type: {dict(df['is_pv'].map({True: 'Particle Verb (PV)', False: 'Noun Compound (NC)'}).value_counts())}")
    print(f"  • Labeled rows with scores: {df['has_label'].sum()} / {len(df)}")
    
    print("\n🔍 Sample 5 Rows:")
    display(df[['context', 'mod', 'head', 'compound', 'lang', 'is_pv', 'mod_avg', 'head_avg']].head())
except Exception as e:
    print(f"⚠️ Labeled files in data/ not found ({e}). Using sample benchmark records for demonstration.")
    raw_rows = [
        {'context': 'We spent the sunny Saturday morning browsing vintage records and lamps at the local flea market.', 'mod': 'flea', 'head': 'market', 'compound': 'flea market', 'lang': 'en', 'is_pv': False, 'mod_avg': 1.4, 'head_avg': 4.8, 'mod_std': 0.45, 'head_std': 0.35, 'has_label': True},
        {'context': 'The legendary actress made her triumphant return to the silver screen after a decade in theater.', 'mod': 'silver', 'head': 'screen', 'compound': 'silver screen', 'lang': 'en', 'is_pv': False, 'mod_avg': 1.15, 'head_avg': 1.8, 'mod_std': 0.62, 'head_std': 0.55, 'has_label': True},
        {'context': 'Nach den anstrengenden Prüfungen hielt sie stolz ihr frisch gedrucktes Abiturzeugnis in den Händen.', 'mod': 'Abitur', 'head': 'Zeugnis', 'compound': 'Abiturzeugnis', 'lang': 'de', 'is_pv': False, 'mod_avg': 4.88, 'head_avg': 4.88, 'mod_std': 0.28, 'head_std': 0.28, 'has_label': True},
        {'context': 'Even after three grueling attempts, the mountaineer refused to give up on reaching the summit.', 'mod': 'give', 'head': 'up', 'compound': 'give up', 'lang': 'en', 'is_pv': True, 'mod_avg': 0.85, 'head_avg': 0.85, 'mod_std': 0.58, 'head_std': 0.58, 'has_label': True},
        {'context': 'Als die Polizei eintraf, haute der Dieb sofort durch die Hintertür ab.', 'mod': 'haute', 'head': 'ab', 'compound': 'abhauen', 'lang': 'de', 'is_pv': True, 'mod_avg': 0.65, 'head_avg': 0.65, 'mod_std': 0.70, 'head_std': 0.70, 'has_label': True},
    ]
    df = pd.DataFrame(raw_rows)
    display(df)

In [ ]:
# =============================================================================
# 4. Subtoken Span Finder & Explicit Role IDs Inspection
# =============================================================================
from transformers import AutoTokenizer
from src.marks import find_spans
from src.data import CompDataset, collate_comp

tokenizer = AutoTokenizer.from_pretrained(cfg.backbone)

print("🔍 Testing Subtoken Span Finder on Noun Compounds & Separable German PVs:\n")
test_cases = [
    ("We bought a rare lamp at the flea market.", "flea", "market", "en"),
    ("In der Schatulle lag ein alter Liebesbrief.", "Liebe", "Brief", "de"),
    ("He decided to give up after three attempts.", "give", "up", "en"),
    ("Der Dieb haute sofort durch die Hintertür ab.", "haute", "ab", "de"),
]

for ctx, mod, head, lang in test_cases:
    enc = tokenizer(ctx, return_offsets_mapping=True)
    offsets = [tuple(o) for o in enc['offset_mapping']]
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'])
    
    spans = find_spans(ctx, offsets, mod, head, "")
    print(f"📝 Context: \"{ctx}\"")
    print(f"   Targets: Mod/Verb = '{mod}', Head/Part = '{head}' (Lang: {lang})")
    print(f"   Aligned Spans: Mod {spans.mod} | Head {spans.head} (Found: {spans.found})")
    if spans.mod.start is not None:
        print(f"   -> Mod Tokens: {tokens[spans.mod.start:spans.mod.end]}")
    if spans.head.start is not None:
        print(f"   -> Head Tokens: {tokens[spans.head.start:spans.head.end]}")
    print("-" * 70)

In [ ]:
# =============================================================================
# 5. Neural Architecture & Forward Pass Shape Verification
# =============================================================================
from torch.utils.data import DataLoader
from src.model import build_model

model = build_model(cfg, device)
print(f"✅ Model '{cfg.model_backend}' successfully instantiated on {device}!")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"📊 Total Parameters: {total_params:,} | Trainable: {trainable_params:,}")

# Create mini-batch and test forward pass
expanded_rows = expand_targets(raw_rows[:4], cfg.targets)
ds = CompDataset(expanded_rows, tokenizer, max_len=cfg.max_context_length)
loader = DataLoader(ds, batch_size=len(ds), collate_fn=collate_comp)
batch = next(iter(loader))

input_ids = batch['input_ids'].to(device)
attention_mask = batch['attention_mask'].to(device)
role_ids = batch['role_ids'].to(device)

with torch.no_grad():
    out = model(input_ids=input_ids, attention_mask=attention_mask, role_ids=role_ids)

print("\n🔬 Forward Pass Tensor Shapes:")
print(f"  • Input IDs:               {input_ids.shape}")
print(f"  • Explicit Role IDs:       {role_ids.shape}")
print(f"  • Cross-Attn Map A:        {out['cross_attn_map'].shape} [B, 3, S]")
print(f"  • Component Matrix (MHSA): {out['comp_matrix'].shape} [B, 3, 3]")
print(f"  • GaussHead mu_all:        {out['mu_all'].shape} [B, 3] (Continuous mu in [0, 1])")
print(f"  • GaussHead sigma_all:     {out['sigma_all'].shape} [B, 3] (Variance spread > 0.04)")

In [ ]:
# =============================================================================
# 6. Full Training Execution with Metrics & Validation Loss
# =============================================================================
import logging
from src.trainer import Trainer

logger = get_logger('kaggle_runner', level=logging.INFO)
output_dir = Path("./output_kaggle")
output_dir.mkdir(parents=True, exist_ok=True)

# Prepare 80/20 train/val split
n_train = int(len(raw_rows) * 0.8)
train_data = raw_rows[:max(n_train, 1)]
val_data = raw_rows[max(n_train, 1):] if len(raw_rows) > 1 else raw_rows

trainer = Trainer(cfg, device, logger, output_dir)
print(f"🚀 Starting 2-Phase Training ({cfg.freeze_epochs} freeze + {cfg.unfreeze_epochs} unfreeze epochs)...")

result = trainer.fit(train_data, val_data, fold=0)

print(f"\n🏆 Best Validation Results (Epoch {result.best_epoch}):")
print(f"  • Spearman Rho (Mod / Verb):   {result.rho_mod:.4f}")
print(f"  • Spearman Rho (Head / Part):  {result.rho_head:.4f}")
print(f"  • Spearman Rho (Particle Verb):{result.rho_pv:.4f}")
print(f"  • Macro-Mean Spearman Rho:     {result.rho_mean:.4f}")

In [ ]:
# =============================================================================
# 7. Interactive Prediction on Custom Sentences
# =============================================================================
@torch.no_grad()
def test_custom_sentence(sentence: str, mod_word: str, head_word: str, lang: str = 'en'):
    model.eval()
    enc = tokenizer(sentence, return_offsets_mapping=True, return_tensors='pt', max_length=cfg.max_context_length, padding='max_length', truncation=True)
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    offsets = [tuple(o) for o in enc['offset_mapping'].squeeze(0).tolist()]
    
    spans = find_spans(sentence, offsets, mod_word, head_word, "")
    role_ids = torch.zeros_like(input_ids)
    if spans.mod.start is not None:
        role_ids[0, spans.mod.start:spans.mod.end] = 1
    if spans.head.start is not None:
        role_ids[0, spans.head.start:spans.head.end] = 2
        
    out = model(input_ids=input_ids, attention_mask=attention_mask, role_ids=role_ids)
    
    # Scale continuous predictions back to [1.0, 5.0]
    score_0 = out['mu_0'].item() * 5.0
    score_1 = out['mu_1'].item() * 5.0
    score_2 = out['mu_2'].item() * 5.0
    
    print(f"\n📝 Sentence: \"{sentence}\"")
    print(f"🎯 Target: Slot 0 = '{mod_word}', Slot 1 = '{head_word}' (Lang: {lang})")
    print(f"📊 Predicted Compositionality Ratings (1.0 -> 5.0):")
    print(f"  • Slot 0 (Mod / Base Verb):   Score = {score_0:.2f}")
    print(f"  • Slot 1 (Head / Particle):   Score = {score_1:.2f}")
    print(f"  • Slot 2 (Whole Expression):  Score = {score_2:.2f}")

# Test cases
test_custom_sentence("We bought a rare antique lamp at the local flea market.", "flea", "market", "en")
test_custom_sentence("Der Dieb haute sofort durch die Hintertür ab.", "haute", "ab", "de")